# divide the datasets into num_clients clients and a test set. Each client has its own valdation set as a train-val split list

In [23]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [24]:
# get the datasets

# omics data
omics_data_c = pd.read_csv(
    os.path.join('src/datasets_csv/raw_rna_data/combine/brca', 'rna_clean.csv'),    # all rna data
    engine='python', 
    index_col=0
)

omics_data_h = pd.read_csv(
    os.path.join('src/datasets_csv/raw_rna_data/hallmarks/brca', 'rna_clean.csv'),    # all rna data
    engine='python', 
    index_col=0
)

omics_data_x = pd.read_csv(
    os.path.join('src/datasets_csv/raw_rna_data/xena/brca', 'rna_clean.csv'),    # all rna data
    engine='python', 
    index_col=0
)

# metadata
label_data = pd.read_csv('src/datasets_csv/metadata/tcga_brca.csv', low_memory=False)    # contains pairing between case_id and slide_id(s) + info on survival

# clinical data
clinical_data = pd.read_csv('src/datasets_csv/clinical_data/tcga_brca_clinical.csv', index_col=0) # clinical data


In [25]:
num_clients = 4
subfolders = ['raw_rna_data/combine/brca', 'raw_rna_data/hallmarks/brca', 'raw_rna_data/xena/brca', 'metadata', 'clinical_data', 'pathway_compositions']

# create the folders and subfolders
for i in range(num_clients):
    client_folder = f'src/clients/client_{i}'
    os.makedirs(client_folder, exist_ok=True)
    
    for subfolder in subfolders:
        os.makedirs(os.path.join(client_folder, subfolder), exist_ok=True)

# create the global test set folder
test_folder = 'src/clients/test_set'
os.makedirs(test_folder, exist_ok=True)
    
for subfolder in subfolders:
    os.makedirs(os.path.join(test_folder, subfolder), exist_ok=True)

# create the global val set folder
#val_folder = 'src/clients/val_set'
#os.makedirs(val_folder, exist_ok=True)
#    
#for subfolder in subfolders:
#    os.makedirs(os.path.join(val_folder, subfolder), exist_ok=True)

In [26]:
# get the 20% of the patients as test set
#  get the common ids between the datasets (the ones in label_data)
common_ids = label_data["case_id"].unique()
np.random.shuffle(common_ids)
print(f"Total number of patients in label_data: {len(common_ids)}")

common_ids, test_ids  = train_test_split(common_ids, test_size=0.15, random_state=42)

# get 20% of remaining common_ids as val test
#common_ids, val_ids  = train_test_split(common_ids, test_size=0.15, random_state=42)

Total number of patients in label_data: 659


In [27]:
common_ids.shape, test_ids.shape#, val_ids.shape

((560,), (99,))

In [28]:
test_df = pd.DataFrame(test_ids, columns=['test'])
test_df.head(5)

# save test set
test_df.to_csv(f'src/clients/test_set/test.csv')

In [29]:
test_df.head(5)

,test
0,TCGA-C8-A26W
1,TCGA-C8-A1HE
2,TCGA-A2-A0CQ
3,TCGA-BH-A0B6
4,TCGA-A1-A0SH


In [30]:
# get the data for the test set and save it
split_df = label_data[label_data["case_id"].isin(test_ids)]
split_df = split_df.drop(columns=['Unnamed: 0'])
split_df = split_df.reset_index(drop=True)
split_df.to_csv(f'src/clients/test_set/metadata/tcga_brca.csv')

split_df = clinical_data[clinical_data["case_id"].isin(test_ids)]
split_df = split_df.reset_index(drop=True)
split_df.to_csv(f'src/clients/test_set/clinical_data/tcga_brca_clinical.csv')

split_df_c = omics_data_c[omics_data_c.index.isin(test_ids)]
split_df_c.to_csv(f'src/clients/test_set/raw_rna_data/combine/brca/rna_clean.csv')

split_df_h = omics_data_h[omics_data_h.index.isin(test_ids)]
split_df_h.to_csv(f'src/clients/test_set/raw_rna_data/hallmarks/brca/rna_clean.csv')

split_df_x = omics_data_x[omics_data_x.index.isin(test_ids)]
split_df_x.to_csv(f'src/clients/test_set/raw_rna_data/xena/brca/rna_clean.csv')

In [31]:
'''
val_df = pd.DataFrame(val_ids, columns=['val'])
val_df.head(5)

# save test set
val_df.to_csv(f'src/clients/val_set/val.csv')

# get the data for the val set and save it
split_df = label_data[label_data["case_id"].isin(val_ids)]
split_df = split_df.drop(columns=['Unnamed: 0'])
split_df = split_df.reset_index(drop=True)
split_df.to_csv(f'src/clients/val_set/metadata/tcga_brca.csv')

split_df = clinical_data[clinical_data["case_id"].isin(val_ids)]
split_df = split_df.reset_index(drop=True)
split_df.to_csv(f'src/clients/val_set/clinical_data/tcga_brca_clinical.csv')

split_df_c = omics_data_c[omics_data_c.index.isin(val_ids)]
split_df_c.to_csv(f'src/clients/val_set/raw_rna_data/combine/brca/rna_clean.csv')

split_df_h = omics_data_h[omics_data_h.index.isin(val_ids)]
split_df_h.to_csv(f'src/clients/val_set/raw_rna_data/hallmarks/brca/rna_clean.csv')

split_df_x = omics_data_x[omics_data_x.index.isin(val_ids)]
split_df_x.to_csv(f'src/clients/val_set/raw_rna_data/xena/brca/rna_clean.csv')'''

'\nval_df = pd.DataFrame(val_ids, columns=[\'val\'])\nval_df.head(5)\n\n# save test set\nval_df.to_csv(f\'src/clients/val_set/val.csv\')\n\n# get the data for the val set and save it\nsplit_df = label_data[label_data["case_id"].isin(val_ids)]\nsplit_df = split_df.drop(columns=[\'Unnamed: 0\'])\nsplit_df = split_df.reset_index(drop=True)\nsplit_df.to_csv(f\'src/clients/val_set/metadata/tcga_brca.csv\')\n\nsplit_df = clinical_data[clinical_data["case_id"].isin(val_ids)]\nsplit_df = split_df.reset_index(drop=True)\nsplit_df.to_csv(f\'src/clients/val_set/clinical_data/tcga_brca_clinical.csv\')\n\nsplit_df_c = omics_data_c[omics_data_c.index.isin(val_ids)]\nsplit_df_c.to_csv(f\'src/clients/val_set/raw_rna_data/combine/brca/rna_clean.csv\')\n\nsplit_df_h = omics_data_h[omics_data_h.index.isin(val_ids)]\nsplit_df_h.to_csv(f\'src/clients/val_set/raw_rna_data/hallmarks/brca/rna_clean.csv\')\n\nsplit_df_x = omics_data_x[omics_data_x.index.isin(val_ids)]\nsplit_df_x.to_csv(f\'src/clients/val_set/

In [32]:
main_size = label_data.shape[0]
main_size

659

In [33]:
# Split into 10 roughly equal groups
ids_splits = np.array_split(common_ids, num_clients)
print([len(split) for split in ids_splits])  # check sizes of the splits

[140, 140, 140, 140]


In [34]:
# save the splits for label_data
for i, split in enumerate(ids_splits):
    split_df = label_data[label_data["case_id"].isin(split)]
    split_df = split_df.drop(columns=['Unnamed: 0'])
    split_df = split_df.reset_index(drop=True)
    split_df.to_csv(f'src/clients/client_{i}/metadata/tcga_brca.csv')

In [35]:
# check:
#ids_splits[0]

In [36]:
# get case_ids from clinical_data that is not in label data
clinical_only_ids = [pid for pid in clinical_data['case_id'].unique() if pid not in common_ids and pid not in test_ids]
print(f"Number of patient_ids in clinical data but not in label data: {len(clinical_only_ids)}")

# get split for this ids
clinical_only_splits = np.array_split(clinical_only_ids, num_clients)
print([len(split) for split in clinical_only_splits])

Number of patient_ids in clinical data but not in label data: 230
[58, 58, 57, 57]


In [37]:
# save the splits for clinical_data
for i, split in enumerate(ids_splits):
    split_df = clinical_data[clinical_data["case_id"].isin(split)]
    split_df = pd.concat([split_df, clinical_data[clinical_data["case_id"].isin(clinical_only_splits[i])]])
    #split_df = split_df.drop(columns=['Unnamed: 0'])
    split_df = split_df.reset_index(drop=True)
    split_df.to_csv(f'src/clients/client_{i}/clinical_data/tcga_brca_clinical.csv')

In [38]:
# get case_ids from omics_data that is not in label data

omics_c_only_ids = [pid for pid in omics_data_c.index.unique() if pid not in common_ids and pid not in test_ids]
print(f"Number of patient_ids in omics data combine but not in label data: {len(omics_c_only_ids)}")

# get split for this ids
omics_c_only_splits = np.array_split(omics_c_only_ids, num_clients)
print([len(split) for split in omics_c_only_splits])


omics_h_only_ids = [pid for pid in omics_data_h.index.unique() if pid not in common_ids and pid not in test_ids]
print(f"Number of patient_ids in omics data hallmarks but not in label data: {len(omics_h_only_ids)}")

# get split for this ids
omics_h_only_splits = np.array_split(omics_h_only_ids, num_clients)
print([len(split) for split in omics_h_only_splits])


omics_x_only_ids = [pid for pid in omics_data_x.index.unique() if pid not in common_ids and pid not in test_ids]
print(f"Number of patient_ids in omics data xena but not in label data: {len(omics_x_only_ids)}")

# get split for this ids
omics_x_only_splits = np.array_split(omics_x_only_ids, num_clients)
print([len(split) for split in omics_x_only_splits])

Number of patient_ids in omics data combine but not in label data: 67
[17, 17, 17, 16]
Number of patient_ids in omics data hallmarks but not in label data: 67
[17, 17, 17, 16]
Number of patient_ids in omics data xena but not in label data: 67
[17, 17, 17, 16]


In [39]:
# save the splits for omics_data
for i, split in enumerate(ids_splits):

    split_df_c = omics_data_c[omics_data_c.index.isin(split)]
    split_df_c = pd.concat([split_df_c, omics_data_c[omics_data_c.index.isin(omics_c_only_splits[i])]])
    split_df_c.to_csv(f'src/clients/client_{i}/raw_rna_data/combine/brca/rna_clean.csv')

    split_df_h = omics_data_h[omics_data_h.index.isin(split)]
    split_df_h = pd.concat([split_df_h, omics_data_h[omics_data_h.index.isin(omics_h_only_splits[i])]])
    split_df_h.to_csv(f'src/clients/client_{i}/raw_rna_data/hallmarks/brca/rna_clean.csv')

    split_df_x = omics_data_x[omics_data_x.index.isin(split)]
    split_df_x = pd.concat([split_df_x, omics_data_x[omics_data_x.index.isin(omics_x_only_splits[i])]])
    split_df_x.to_csv(f'src/clients/client_{i}/raw_rna_data/xena/brca/rna_clean.csv')

now add the extra data, the datasets that are not related to patients to the clients

In [40]:
datasets_meta = ['combine_signatures.csv', 'hallmarks_signatures.csv', 'xena_signatures.csv', 'signatures.csv']
datasets_path = ['combine_comps.csv', 'hallmarks_comps.csv', 'xena_comps.csv']


for i in range(num_clients):
    client_folder = f'src/clients/client_{i}'

    # metadata
    for dataset in datasets_meta:
        dataset_path = os.path.join('src/datasets_csv', 'metadata', dataset)
        df = pd.read_csv(dataset_path, index_col=0)
        df.to_csv(os.path.join(client_folder, 'metadata', dataset))
    
    # pathway compositions
    for dataset in datasets_path:
        dataset_path = os.path.join('src/datasets_csv', 'pathway_compositions', dataset)
        df = pd.read_csv(dataset_path, index_col=0)
        df.to_csv(os.path.join(client_folder, 'pathway_compositions', dataset))


In [41]:
import os
import pandas as pd
datasets_meta = ['combine_signatures.csv', 'hallmarks_signatures.csv', 'xena_signatures.csv', 'signatures.csv']
datasets_path = ['combine_comps.csv', 'hallmarks_comps.csv', 'xena_comps.csv']
client_folder = f'src/clients/test_set'

# metadata
for dataset in datasets_meta:
    dataset_path = os.path.join('src/datasets_csv', 'metadata', dataset)
    df = pd.read_csv(dataset_path, index_col=0)
    df.to_csv(os.path.join(client_folder, 'metadata', dataset))

# pathway compositions
for dataset in datasets_path:
    dataset_path = os.path.join('src/datasets_csv', 'pathway_compositions', dataset)
    df = pd.read_csv(dataset_path, index_col=0)
    df.to_csv(os.path.join(client_folder, 'pathway_compositions', dataset))

'''
client_folder = f'src/clients/val_set'

# metadata
for dataset in datasets_meta:
    dataset_path = os.path.join('src/datasets_csv', 'metadata', dataset)
    df = pd.read_csv(dataset_path, index_col=0)
    df.to_csv(os.path.join(client_folder, 'metadata', dataset))

# pathway compositions
for dataset in datasets_path:
    dataset_path = os.path.join('src/datasets_csv', 'pathway_compositions', dataset)
    df = pd.read_csv(dataset_path, index_col=0)
    df.to_csv(os.path.join(client_folder, 'pathway_compositions', dataset))'''

"\nclient_folder = f'src/clients/val_set'\n\n# metadata\nfor dataset in datasets_meta:\n    dataset_path = os.path.join('src/datasets_csv', 'metadata', dataset)\n    df = pd.read_csv(dataset_path, index_col=0)\n    df.to_csv(os.path.join(client_folder, 'metadata', dataset))\n\n# pathway compositions\nfor dataset in datasets_path:\n    dataset_path = os.path.join('src/datasets_csv', 'pathway_compositions', dataset)\n    df = pd.read_csv(dataset_path, index_col=0)\n    df.to_csv(os.path.join(client_folder, 'pathway_compositions', dataset))"

Now create a validation set for all clients

In [42]:
from random import shuffle

for i in range(num_clients):
    client_dir = f'src/clients/client_{i}'

    label_ids = ids_splits[i]
    clinical_ids = clinical_only_splits[i]
    omics_ids = omics_c_only_splits[i]

    all_ids = []
    all_ids.extend(label_ids)
    all_ids.extend(clinical_ids)
    all_ids.extend(omics_ids)
    print(f'Client {i}, size of all ids: {len(all_ids)}')

    shuffle(all_ids)
    train_ids, val_ids = train_test_split(all_ids, test_size = 0.2, random_state=42)
    print(f'train ids: {len(train_ids)};   val ids: {len(val_ids)}')

    data = {'train': train_ids, 'val': val_ids}
    df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in data.items()]))
    print(df.head(3))

    df.to_csv(f'src/clients/client_{i}/train-val_split.csv')


Client 0, size of all ids: 215
train ids: 172;   val ids: 43
          train           val
0  TCGA-A7-A4SD  TCGA-AR-A24W
1  TCGA-E2-A109  TCGA-A7-A4SC
2  TCGA-BH-A0DE  TCGA-A2-A3KD
Client 1, size of all ids: 215
train ids: 172;   val ids: 43
          train           val
0  TCGA-BH-A0HI  TCGA-AC-A2QI
1  TCGA-A8-A095  TCGA-AR-A256
2  TCGA-C8-A278  TCGA-E2-A1BD
Client 2, size of all ids: 214
train ids: 171;   val ids: 43
          train           val
0  TCGA-D8-A27W  TCGA-E9-A5FL
1  TCGA-AR-A24P  TCGA-E2-A1IH
2  TCGA-AR-A0U2  TCGA-A2-A04Y
Client 3, size of all ids: 213
train ids: 170;   val ids: 43
          train           val
0  TCGA-A2-A0T2  TCGA-A2-A0ES
1  TCGA-GM-A3XN  TCGA-B6-A0X5
2  TCGA-AR-A252  TCGA-E9-A1N6
